# AWF — Compress & Chat with Any LLM

**Two compression paths, both real. Pick the one that fits your use case.**

## Two paths in this notebook

### Path A — Standalone GGUF Q4_K_M (RECOMMENDED)
- Output: one `.gguf` file, ~35% of original size
- Runtime: loads ONLY the .gguf, no PyTorch / Transformers needed
- Verified with a real [10K-conversation test](../gguf_standalone/docs/RESULTS.md) — 100% success rate
- Works with Ollama, LM Studio, llama.cpp, kobold.cpp, etc.
- Best for: shipping models to users, edge deployment, production

### Path B — AWF SVD + int8 (in-process)
- Output: a `.pt` checkpoint with SVD-compressed weight matrices
- Runtime: loads via PyTorch, reconstructs weights to fp32 in memory
- Best for: research, experimentation, when you need PyTorch gradients

## When to use which

| If you want... | Use |
|---|---|
| Smallest file to ship to users | **Path A** (GGUF Q4_K_M) |
| Real runtime memory savings | **Path A** (GGUF Q4_K_M) |
| Ollama / LM Studio compatibility | **Path A** (GGUF Q4_K_M) |
| To fine-tune the compressed model | **Path B** (AWF SVD) |
| To study the SVD reconstruction | **Path B** (AWF SVD) |

## Documentation

- **Standalone GGUF module**: [`gguf_standalone/`](../gguf_standalone/) — README, USAGE, ARCHITECTURE, RESULTS, COMPRESSION_LEVELS, OLLAMA
- **Compression level comparison** (Q8_0 vs Q5_K_M vs Q4_K_M vs Q2_K, with perplexity + sample outputs): [`gguf_standalone/docs/COMPRESSION_LEVELS.md`](../gguf_standalone/docs/COMPRESSION_LEVELS.md)
- **Using compressed models with Ollama**: [`gguf_standalone/docs/OLLAMA.md`](../gguf_standalone/docs/OLLAMA.md)
- **10K conversation test results** (proof the compressed file works standalone): [`gguf_standalone/docs/RESULTS.md`](../gguf_standalone/docs/RESULTS.md)

## Supported source models

| Model | Type | Params | Path A size | Path B size | Token? |
|-------|------|--------|-------------|-------------|--------|
| Qwen/Qwen2.5-0.5B-Instruct | Chat | 0.5B | ~380 MB | ~250 MB | No |
| Qwen/Qwen2-1.5B-Instruct | Chat | 1.5B | ~1.1 GB | ~600 MB | No |
| microsoft/Phi-3-mini-4k-instruct | Chat | 3.8B | ~2.3 GB | ~1.2 GB | No |
| THUDM/glm-4-9b-chat | Chat | 9B | ~5.5 GB | ~3 GB | No |
| deepseek-ai/deepseek-llm-7b-chat | Chat | 7B | ~4.3 GB | ~2.5 GB | No |
| mistralai/Mistral-7B-Instruct-v0.3 | Chat | 7B | ~4.3 GB | ~2.5 GB | Yes |
| Qwen/Qwen2-7B-Instruct | Chat | 7B | ~4.3 GB | ~2.5 GB | No |
| meta-llama/Llama-3.1-8B-Instruct | Chat | 8B | ~5 GB | ~2.8 GB | Yes |

## Time
- Small models (0.5-1.5B): 5-10 minutes on Colab (CPU is fine for Path A)
- Medium models (7-9B): 15-30 minutes on Colab T4 GPU


In [ ]:
# @title Setup (shared between Path A and Path B)
!git clone https://github.com/Deexv/AWF.git 2>/dev/null || true
%cd AWF
!pip install -q -r requirements.txt transformers accelerate

# Path A also needs llama-cpp-python and the llama.cpp convert+quantize binaries.
# We install llama-cpp-python here; the notebook builds llama.cpp in the Path A cell.
!pip install -q llama-cpp-python

import torch, sys, os
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import llama_cpp
print(f'llama-cpp-python: {llama_cpp.__version__}')


# Path A — Standalone GGUF Q4_K_M (RECOMMENDED)

This path produces a single `.gguf` file that loads on its own — no PyTorch, no Transformers, no HuggingFace cache. Just the one file.

**Verified**: A [10,000-conversation test](../gguf_standalone/docs/RESULTS.md) on Qwen2.5-0.5B-Instruct-Q4_K_M.gguf achieved 100% success rate, ~20 tok/s on a 2-core CPU.

## Step A1: Build llama.cpp (one-time, ~2 min on Colab)

We need `convert_hf_to_gguf.py` and `llama-quantize` from llama.cpp.


In [ ]:
# @title Build llama.cpp (one-time per Colab session)
import os
os.chdir('/content')
if not os.path.exists('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
os.chdir('/content/llama.cpp')
!pip install -q -r requirements/requirements-convert_hf_to_gguf.txt
!apt-get install -y cmake build-essential 2>&1 | tail -2
!mkdir -p build && cd build && cmake .. -DGGML_NATIVE=ON -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_APP=OFF 2>&1 | tail -2
!cd build && cmake --build . --target llama-quantize -j 2>&1 | tail -3
print('Built: /content/llama.cpp/build/bin/llama-quantize')


## Step A2: Pick a model and quantization level

See [`gguf_standalone/docs/COMPRESSION_LEVELS.md`](../gguf_standalone/docs/COMPRESSION_LEVELS.md) for the full quality/size tradeoff:

| Quant | Size (% of F16) | PPL on wikitext-2 | Quality |
|---|---|---|---|
| Q8_0 | 51% | 12.49 | Effectively lossless |
| Q5_K_M | 41% | 12.99 | Indistinguishable from Q8_0 |
| **Q4_K_M** | **38%** | **12.76** | **Recommended default** |
| Q2_K | 33% | 15.85 | Visible quality drop on hard prompts |


In [ ]:
# @title Configuration
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # @param ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2-1.5B-Instruct', 'microsoft/Phi-3-mini-4k-instruct', 'THUDM/glm-4-9b-chat', 'deepseek-ai/deepseek-llm-7b-chat', 'Qwen/Qwen2-7B-Instruct', 'mistralai/Mistral-7B-Instruct-v0.3', 'meta-llama/Llama-3.1-8B-Instruct']
QUANT_TYPE = 'Q4_K_M'  # @param ['Q8_0', 'Q6_K', 'Q5_K_M', 'Q4_K_M', 'Q3_K_M', 'Q2_K']

MODEL_SHORT = MODEL_ID.split('/')[-1]
HF_DIR = f'/content/models/{MODEL_SHORT}'
F16_GGUF = f'/content/{MODEL_SHORT}-F16.gguf'
COMPRESSED_GGUF = f'/content/{MODEL_SHORT}-{QUANT_TYPE}.gguf'

print(f'Model:        {MODEL_ID}')
print(f'Quant type:   {QUANT_TYPE}')
print(f'HF dir:       {HF_DIR}')
print(f'F16 (intermediate, will be deleted): {F16_GGUF}')
print(f'Compressed (final, standalone):     {COMPRESSED_GGUF}')


## Step A3: Download the original HF model

This is the only step that touches the uncompressed model. After Step A4 produces the compressed GGUF, we delete this directory to prove the compressed file is standalone.


In [ ]:
# @title Download HF model
from huggingface_hub import snapshot_download
import os
os.makedirs('/content/models', exist_ok=True)
print(f'Downloading {MODEL_ID} ...')
local_dir = snapshot_download(repo_id=MODEL_ID, local_dir=HF_DIR)
print(f'Saved to: {local_dir}')

total = 0
for root, _, files in os.walk(HF_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f'Original size: {total/1024/1024:.1f} MiB ({total/1024/1024/1024:.2f} GiB)')


## Step A4: Convert HF → F16 GGUF, then quantize to {QUANT_TYPE}

Two-step process:
1. Lossless conversion from HF safetensors to F16 GGUF (llama.cpp's container format)
2. Quantize F16 → the chosen quant type (this is the actual compression)


In [ ]:
# @title Convert + quantize
import os
os.chdir('/content/llama.cpp')

# Step 1: HF → F16 GGUF (lossless)
print('--- Step 1: HF → F16 GGUF (lossless) ---')
!python convert_hf_to_gguf.py "{HF_DIR}" --outfile "{F16_GGUF}" --outtype f16
f16_size = os.path.getsize(F16_GGUF)
print(f'F16 GGUF: {f16_size/1024/1024:.1f} MiB')

# Step 2: Quantize F16 → chosen quant type (the actual compression)
print(f'\n--- Step 2: F16 → {QUANT_TYPE} (the actual compression) ---')
!./build/bin/llama-quantize "{F16_GGUF}" "{COMPRESSED_GGUF}" {QUANT_TYPE}
comp_size = os.path.getsize(COMPRESSED_GGUF)
print(f'\nCompressed ({QUANT_TYPE}) GGUF: {comp_size/1024/1024:.1f} MiB')
print(f'Compression ratio: {comp_size/f16_size*100:.1f}% of F16 (saved {100 - comp_size/f16_size*100:.1f}%)')


## Step A5: Delete the originals — the compressed file is standalone

This is the proof. After this cell, the ONLY model file on disk is the compressed `.gguf`. If the next cell (chat) works, the compression is genuinely standalone.


In [ ]:
# @title Delete originals (proves the .gguf is standalone)
import shutil, os
if os.path.exists(HF_DIR):
    shutil.rmtree(HF_DIR)
    print(f'Deleted: {HF_DIR}')
if os.path.exists(F16_GGUF):
    os.remove(F16_GGUF)
    print(f'Deleted: {F16_GGUF}')

print(f'\nRemaining model file: {COMPRESSED_GGUF}')
print(f'Size: {os.path.getsize(COMPRESSED_GGUF)/1024/1024:.1f} MiB')
print(f'\nOnly the compressed .gguf exists. No base model anywhere.')


## Step A6: Chat with the compressed model

Loads ONLY the `.gguf` file via `llama-cpp-python`. No PyTorch, no Transformers, no HF cache.


In [ ]:
# @title Chat with the compressed .gguf
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=COMPRESSED_GGUF,
    n_ctx=2048,
    n_threads=2,
    n_gpu_layers=99 if torch.cuda.is_available() else 0,  # use GPU if available
    verbose=False,
)
print(f'Loaded compressed model in {time.time()-t0:.2f}s')
print(f'Model size on disk: {os.path.getsize(COMPRESSED_GGUF)/1024/1024:.1f} MiB')
print(f'(No base model loaded. Only the .gguf file is open.)\n')

# Quick smoke test
prompt = 'What is the capital of France?'
out = llm.create_chat_completion(
    messages=[
        {'role': 'system', 'content': 'You are a helpful, concise assistant.'},
        {'role': 'user', 'content': prompt},
    ],
    max_tokens=64,
    temperature=0.7,
)
print(f'You: {prompt}')
print(f'AI:  {out["choices"][0]["message"]["content"]}')
print(f'Usage: {out.get("usage")}')


In [ ]:
# @title Try multiple prompts
prompts = [
    'What is the capital of France?',
    'Write a short poem about autumn.',
    'Explain how a computer works to a 5-year-old.',
    'What are 3 tips for staying healthy?',
]

for p in prompts:
    print(f'\n{"="*60}')
    print(f'You: {p}')
    out = llm.create_chat_completion(
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'user', 'content': p},
        ],
        max_tokens=150,
        temperature=0.7,
    )
    print(f'AI:  {out["choices"][0]["message"]["content"]}')


## Step A7: Export for Ollama

The compressed `.gguf` works directly with Ollama. See [`gguf_standalone/docs/OLLAMA.md`](../gguf_standalone/docs/OLLAMA.md) for full instructions.


In [ ]:
# @title Generate Modelfile for Ollama
import os

# Auto-detect chat template family from model ID
model_lower = MODEL_ID.lower()
if 'qwen' in model_lower:
    template = '''{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
'''
    stops = ['<|im_start|>', '<|im_end|>', '</s>']
elif 'llama-3' in model_lower or 'llama3' in model_lower:
    template = '''<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>'''
    stops = ['<|eot_id|>', '<|start_header_id|>']
elif 'mistral' in model_lower:
    template = '''[INST] {{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }} [/INST] {{ .Response }}'''
    stops = ['[INST]', '[/INST]']
else:
    # Fallback: use the GGUF's baked-in template (Ollama reads it from metadata)
    template = None
    stops = []

modelfile_path = f'/content/Modelfile.{MODEL_SHORT}'
with open(modelfile_path, 'w') as f:
    f.write(f'FROM {COMPRESSED_GGUF}\n\n')
    if template:
        f.write(f'TEMPLATE """{template}"""\n\n')
    for s in stops:
        f.write(f'PARAMETER stop "{s}"\n')
    f.write('PARAMETER temperature 0.7\n')
    f.write('PARAMETER top_p 0.9\n')

print(f'Wrote: {modelfile_path}')
print('--- Modelfile contents ---')
print(open(modelfile_path).read())
print('---')
print(f'\nTo use with Ollama:')
print(f'  1. Install Ollama: https://ollama.ai')
print(f'  2. ollama create {MODEL_SHORT.lower()}-{QUANT_TYPE.lower()} -f {modelfile_path}')
print(f'  3. ollama run {MODEL_SHORT.lower()}-{QUANT_TYPE.lower()}')
print(f'\nFull instructions: gguf_standalone/docs/OLLAMA.md')


## Step A8: Save the compressed file to Google Drive

The compressed `.gguf` is small enough to fit on a free Google Drive tier for most models.


In [ ]:
# @title Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF_GGUF
save_path = f'/content/drive/MyDrive/AWF_GGUF/{MODEL_SHORT}-{QUANT_TYPE}.gguf'
!cp "{COMPRESSED_GGUF}" "{save_path}"
print(f'Saved to: {save_path}')
print(f'Size: {os.path.getsize(save_path)/1024/1024:.1f} MiB')
print(f'\nTo use in a new session:')
print(f'  !cp "{save_path}" /content/')
print(f'  Then load with llama-cpp-python (see Step A6)')


---

# Path B — AWF SVD + int8 (in-process)

This is the original AWF compression path. Output is a `.pt` checkpoint with
SVD-compressed weight matrices. Loads via PyTorch, reconstructs weights to fp32
in memory.

**Use this path if you need to fine-tune the compressed model, or study the SVD
reconstruction.** For shipping to users / running in production, use Path A above.

## Step B1: List Available Models


In [ ]:
# @title List AWF-supported models
!python scripts/chat_real.py --list


## Step B2: Compress with AWF (SVD + int8)

The `keep_ratio` parameter controls how much of the original SVD information is
preserved. Higher = better quality, larger file. See
[`gguf_standalone/docs/COMPRESSION_LEVELS.md`](../gguf_standalone/docs/COMPRESSION_LEVELS.md)
for how this maps to GGUF quant levels.


In [ ]:
# @title AWF compress and save
model_name = 'Qwen/Qwen2-1.5B-Instruct'  # @param ['Qwen/Qwen2-1.5B-Instruct', 'microsoft/Phi-3-mini-4k-instruct', 'THUDM/glm-4-9b-chat', 'deepseek-ai/deepseek-llm-7b-chat', 'distilgpt2', 'gpt2']
keep_ratio = 0.85  # @param {type:'slider', min:0.5, max:0.95, step:0.05}

import torch, sys, os
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
from chat_real import compress_and_store, save_compressed_model
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32, trust_remote_code=True).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
orig_size = n_params * 4 / 1024 / 1024
print(f'Original: {n_params:,} params ({n_params/1e6:.1f}M), {orig_size:.1f} MB')

print(f'\nCompressing (keep_ratio={keep_ratio})...')
compressed_data, ratio = compress_and_store(model, keep_ratio=keep_ratio)

save_name = model_name.replace('/', '_') + '_compressed.pt'
save_path = f'checkpoints/{save_name}'
actual_size = save_compressed_model(model, compressed_data, model_name, save_path)
actual_mb = actual_size / 1024 / 1024

print(f'\nSaved to: {save_path}')
print(f'Original size: {orig_size:.1f} MB')
print(f'Compressed file: {actual_mb:.1f} MB')
print(f'ACTUAL compression: {orig_size / actual_mb:.1f}x')
print(f'RAM needed at runtime: ~{orig_size / 1024:.1f} GB (weights reconstructed to fp32)')
print(f'\nNOTE: For real runtime memory savings, use Path A (GGUF) above.')


## Step B3: Chat with the AWF-compressed model

Uses the in-memory reconstructed model (PyTorch fp32). Note: this loads the
**full fp32** weights into RAM, not the compressed file. To get actual
runtime memory savings, use Path A (GGUF Q4_K_M) instead.


In [ ]:
# @title Generate from AWF-compressed model
prompt = 'What is the capital of France?'  # @param {type:'string'}
max_tokens = 200  # @param {type:'integer'}

is_instruct = 'instruct' in model_name.lower() or 'chat' in model_name.lower()

if is_instruct and hasattr(tokenizer, 'apply_chat_template'):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer.encode(text, return_tensors='pt').to(device)
else:
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=max_tokens,
                            temperature=0.7, top_k=50, do_sample=True,
                            pad_token_id=tokenizer.eos_token_id or 0,
                            repetition_penalty=1.2)

response = tokenizer.decode(output[0], skip_special_tokens=True)

if is_instruct:
    for marker in ['<|im_start|>assistant', 'assistant']:
        if marker in response:
            response = response.split(marker)[-1].strip()
            for end_marker in ['<|im_end|>', '<|end|>', '</s>']:
                if end_marker in response:
                    response = response.split(end_marker)[0].strip()
            break
else:
    if response.startswith(prompt):
        response = response[len(prompt):].strip()

print(response)


## Step B4: Interactive chat (AWF path)

Type your own prompts. (For the GGUF path, see Step A6 above.)


In [ ]:
# @title Interactive chat with AWF model
print('=' * 50)
print(f'  Chat with AWF-compressed {model_name}')
print(f'  Type quit to exit')
print('=' * 50)

while True:
    user = input('\nYou> ').strip()
    if user.lower() in ('quit', 'exit', 'q'):
        break
    if not user:
        continue

    if is_instruct and hasattr(tokenizer, 'apply_chat_template'):
        messages = [{'role': 'user', 'content': user}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = tokenizer.encode(text, return_tensors='pt').to(device)
    else:
        input_ids = tokenizer.encode(user, return_tensors='pt').to(device)

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=200,
                                temperature=0.7, top_k=50, do_sample=True,
                                pad_token_id=tokenizer.eos_token_id or 0,
                                repetition_penalty=1.2)

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    if is_instruct:
        for marker in ['<|im_start|>assistant', 'assistant']:
            if marker in response:
                response = response.split(marker)[-1].strip()
                for end_marker in ['<|im_end|>', '<|end|>', '</s>']:
                    if end_marker in response:
                        response = response.split(end_marker)[0].strip()
                break
    else:
        if response.startswith(user):
            response = response[len(user):].strip()

    print(f'AI> {response}')


## Step B5: Save AWF checkpoint to Google Drive


In [ ]:
# @title Save AWF .pt to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF
!cp {save_path} /content/drive/MyDrive/AWF/
saved_name = save_path.split('/')[-1]
print(f'Saved to Google Drive: /content/drive/MyDrive/AWF/{saved_name}')
print(f'\nTo restore in a new session:')
print(f'  !cp /content/drive/MyDrive/AWF/{saved_name} checkpoints/')
print(f'  Then: python scripts/chat_real.py --load checkpoints/{saved_name}')


---

# Next steps

## Documentation

- **Standalone GGUF module**: [`gguf_standalone/`](../gguf_standalone/) — README, USAGE, ARCHITECTURE, RESULTS, COMPRESSION_LEVELS, OLLAMA
- **Compression level comparison** (Q8_0 vs Q5_K_M vs Q4_K_M vs Q2_K, with perplexity + sample outputs): [`gguf_standalone/docs/COMPRESSION_LEVELS.md`](../gguf_standalone/docs/COMPRESSION_LEVELS.md)
- **Using compressed models with Ollama**: [`gguf_standalone/docs/OLLAMA.md`](../gguf_standalone/docs/OLLAMA.md)
- **10K conversation test results**: [`gguf_standalone/docs/RESULTS.md`](../gguf_standalone/docs/RESULTS.md)


The companion now supports `.gguf` files directly:

```bash
# Direct GGUF (uses llama-cpp-python)

# Via Ollama (uses Ollama's HTTP API)
```


## Run the 10K conversation test yourself

To verify a compressed model works at scale:

```bash
cd gguf_standalone/
python scripts/run_chunks.py --model compressed/your-model-Q4_K_M.gguf \
    --chunks 20 --chunk-size 500 --max-tokens 24
python scripts/finalize_results.py
cat results/final_summary.json
```

## Try different models

Change `MODEL_ID` in Step A2 and re-run Path A. The same code works for any
HuggingFace instruct model that llama.cpp supports (Llama, Mistral, Qwen,
Phi, GLM, DeepSeek, Gemma, etc.).
